# Prosthetics Data Snippets

Small patterns you can copy into the participant workbook if you get stuck. Keep the parts you need and adapt the field names to the task.

In [ ]:
import pandas as pd

df = pd.read_csv('X.csv')
print(df.head())
print(df.info())
print(df.isna().sum())

In [ ]:
import re

CURRENT_YEAR = 2026

def parse_age_input(value):
    if pd.isna(value):
        return None
    # Handle actual numeric values (int or float) before string processing —
    # str(28.0) is '28.0' which fails isdigit() and joins to '280' via digit extraction.
    if isinstance(value, (int, float)):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())
    if numeric_value < 100:
        return numeric_value           # already an age
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value   # birth year
    return numeric_value

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)

In [ ]:
import re

def parse_implant_date(raw):
    if pd.isna(raw):
        return None
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())
    # Year-first formats (YYYY-MM-DD or YYYY/MM/DD)
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return None

df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()

In [ ]:
import hashlib

def anonymize_name(value):
    normalized = str(value).strip().lower()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

# Overwrite Full_Name in-place so the hashed value is in the right column for the join.
df['Full_Name'] = df['Full_Name'].apply(anonymize_name)

# Join to device inventory after masking — use the actual model IDs from the dataset.
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite',   'Cost_USD': 950,  'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha',  'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro',    'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field',  'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}
device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]